In [1]:
#Importa as bibliotecas necessárias
import pandas as pd 
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Importa as tabelas de churn sem duplicado + resolução de tempo modal
df_resolution_time = pd.read_csv('/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/tempo-resolucao-modal-def.csv')
df_accounts_churn = pd.read_csv('/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/criacao-contas-churn-sem-duplicados-prov.csv')

In [4]:
#Deleta a coluna Unnamed da planilha, que foi criada após fazermos uma nova 
#versão do arquivo
del df_accounts_churn['Unnamed: 0']

In [5]:
df_accounts_churn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152224 entries, 0 to 152223
Data columns (total 9 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   ID                    152224 non-null  int64 
 1   FIRST_NAME            152205 non-null  object
 2   GENDER                152224 non-null  object
 3   CITY                  152224 non-null  object
 4   SK.CREATED_AT::DATE   152224 non-null  object
 5   TRANSPORT_MEDIA_TYPE  152224 non-null  object
 6   CARTAO                152207 non-null  object
 7   LEVEL_NAME            152224 non-null  object
 8   FECHA_ULT             152224 non-null  object
dtypes: int64(1), object(8)
memory usage: 10.5+ MB


In [6]:
df_resolution_time

,TICKET_ID,STOREKEEPER_ID,LEVEL_NAME,TRANSPORT_MEDIA_TYPE,SENT_DATA,SENT_HOUR,RESPONSE_AT,RESPONSE_TIME,RESOLUTION_TIME,RESOLUTION_TIME_BUCKET,CITY
0,60e3ba2408e0cf55c8cc8e68,1224698,bronze,bicycle,2021-07-05,23:04:20.917,2021-07-06T02:48:20.283Z,13440,13440.0,(3) Between 2 and 5 hours,Recife
1,61867d939acfd749cfdd3e89,1060149,bronze,car,2021-11-06,10:05:23.781,2021-11-06T10:08:11.863Z,168,87322.0,(6) Between 24 and 72 hours,São José dos Campos
2,61867d939acfd749cfdd3e89,1060149,bronze,car,2021-11-07,10:13:58.893,2021-11-07T10:20:42.453Z,404,87322.0,(6) Between 24 and 72 hours,São José dos Campos
3,61d86d69884df5af1f862d71,230812,bronze,bicycle,2022-01-07,13:51:11.988,2022-01-07T13:54:40.303Z,209,1476.0,(1) Less than 1 hour,Grande São Paulo
4,61d86d69884df5af1f862d71,230812,bronze,bicycle,2022-01-07,13:55:37.808,2022-01-07T14:06:16.658Z,639,1476.0,(1) Less than 1 hour,Grande São Paulo
...,...,...,...,...,...,...,...,...,...,...,...
1888997,624e3fd95e83d47d17706624,72174,bronze,bicycle,2022-04-06,22:35:22.671,2022-04-07T00:51:27.672Z,8165,339421.0,(7) More than 72 hours,Unidentified
1888998,616189c1af659e20da721a23,1363623,bronze,motorbike,2021-10-09,09:23:30.167,2021-10-09T14:45:47.829Z,19337,19354.0,(4) Between 5 and 10 hours,Grande São Paulo
1888999,6286451e1cc6b08ffec6521f,1471813,rookie,car,2022-05-19,12:37:26.415,2022-05-19T15:31:48.577Z,10462,18422.0,(4) Between 5 and 10 hours,Rio De Janeiro
1889000,6286451e1cc6b08ffec6521f,1471813,rookie,car,2022-05-19,10:24:47.326,2022-05-19T11:46:07.359Z,4880,18422.0,(4) Between 5 and 10 hours,Rio De Janeiro


In [7]:
#Remove as colunas inconvenientes para a predição
df_resolution_time = df_resolution_time.drop(columns=[ 'SENT_DATA', 'SENT_HOUR', 'RESPONSE_AT',
                                                      'CITY' ,'TRANSPORT_MEDIA_TYPE',
                                                      'RESPONSE_TIME', 'RESOLUTION_TIME_BUCKET'])

In [8]:
df_resolution_time.rename(columns={'STOREKEEPER_ID': 'ID'}, inplace = True)

In [9]:
df_resolution_time.head()

,TICKET_ID,ID,LEVEL_NAME,RESOLUTION_TIME
0,60e3ba2408e0cf55c8cc8e68,1224698,bronze,13440.0
1,61867d939acfd749cfdd3e89,1060149,bronze,87322.0
2,61867d939acfd749cfdd3e89,1060149,bronze,87322.0
3,61d86d69884df5af1f862d71,230812,bronze,1476.0
4,61d86d69884df5af1f862d71,230812,bronze,1476.0


In [10]:
df_resolution_time = df_resolution_time.sort_values('RESOLUTION_TIME', ascending=False).drop_duplicates('TICKET_ID').sort_index()

In [11]:
df_resolution_time

,TICKET_ID,ID,LEVEL_NAME,RESOLUTION_TIME
0,60e3ba2408e0cf55c8cc8e68,1224698,bronze,13440.0
1,61867d939acfd749cfdd3e89,1060149,bronze,87322.0
3,61d86d69884df5af1f862d71,230812,bronze,1476.0
6,5e5eb403d82c740012ddb2bf,346188,bronze,NaN
7,6183e66f07b89c43301e2ae5,722296,bronze,700010.0
...,...,...,...,...
1888996,624f0b65de0f5279695dce64,546229,bronze,195328.0
1888997,624e3fd95e83d47d17706624,72174,bronze,339421.0
1888998,616189c1af659e20da721a23,1363623,bronze,19354.0
1888999,6286451e1cc6b08ffec6521f,1471813,rookie,18422.0


In [12]:
df_resolution_time

,TICKET_ID,ID,LEVEL_NAME,RESOLUTION_TIME
0,60e3ba2408e0cf55c8cc8e68,1224698,bronze,13440.0
1,61867d939acfd749cfdd3e89,1060149,bronze,87322.0
3,61d86d69884df5af1f862d71,230812,bronze,1476.0
6,5e5eb403d82c740012ddb2bf,346188,bronze,NaN
7,6183e66f07b89c43301e2ae5,722296,bronze,700010.0
...,...,...,...,...
1888996,624f0b65de0f5279695dce64,546229,bronze,195328.0
1888997,624e3fd95e83d47d17706624,72174,bronze,339421.0
1888998,616189c1af659e20da721a23,1363623,bronze,19354.0
1888999,6286451e1cc6b08ffec6521f,1471813,rookie,18422.0


In [13]:
df_feature = df_resolution_time.dropna()

In [14]:
df_feature['RESOLUTION_TIME'] = df_feature['RESOLUTION_TIME'].div(3600)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


In [15]:
pd.options.display.float_format = '{:.2f}'.format

In [16]:
df_feature

,TICKET_ID,ID,LEVEL_NAME,RESOLUTION_TIME
0,60e3ba2408e0cf55c8cc8e68,1224698,bronze,3.73
1,61867d939acfd749cfdd3e89,1060149,bronze,24.26
3,61d86d69884df5af1f862d71,230812,bronze,0.41
7,6183e66f07b89c43301e2ae5,722296,bronze,194.45
9,5f499044185ee7001a93eb82,863458,rookie,0.00
...,...,...,...,...
1888996,624f0b65de0f5279695dce64,546229,bronze,54.26
1888997,624e3fd95e83d47d17706624,72174,bronze,94.28
1888998,616189c1af659e20da721a23,1363623,bronze,5.38
1888999,6286451e1cc6b08ffec6521f,1471813,rookie,5.12


In [17]:
test = df_feature.groupby('ID').sum()

In [18]:
test

,RESOLUTION_TIME
ID,
32825,6138.08
33051,403.03
33188,65.56
33189,3064.95
33197,6428.95
...,...
1542755,50.01
1542810,14.64
1542869,19.26


In [19]:
test2 = df_feature.groupby('ID').mean()

In [20]:
test2

,RESOLUTION_TIME
ID,
32825,1023.01
33051,31.00
33188,8.20
33189,170.28
33197,214.30
...,...
1542755,7.14
1542810,3.66
1542869,6.42


In [21]:
test.rename(columns={'RESOLUTION_TIME': 'RES_TIME_TOTAL'}, inplace = True)

In [22]:
test2.rename(columns={'RESOLUTION_TIME': 'RES_TIME_MEAN'}, inplace = True)

In [23]:
#df_feature = pd.merge(test, test2, on = 'ID')

In [24]:
df_feature['TOTAL_TICKETS'] = df_feature['ID'].map(df_feature['ID'].value_counts())

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


In [25]:
df_feature = df_feature.drop(columns=['TICKET_ID', 'RESOLUTION_TIME'])

In [26]:
df_feature = df_feature.drop_duplicates()

In [27]:
#df_feature = pd.merge(test, test2, on = 'ID')

In [28]:
df_fe = pd.merge(df_feature, test, on= 'ID')
print(df_fe)

            ID LEVEL_NAME  TOTAL_TICKETS  RES_TIME_TOTAL
0      1224698     bronze             12         2025.32
1      1060149     bronze             11         6549.40
2       230812     bronze             29          333.09
3       722296     bronze             10          407.96
4       863458     rookie             22        16199.85
...        ...        ...            ...             ...
97820  1420787     rookie              1           86.53
97821  1508698     rookie              1           52.45
97822   471172     bronze              1           57.32
97823  1432270     bronze              1           43.93
97824  1499730     silver              1          125.89

[97825 rows x 4 columns]


In [29]:
df_fe_resolution_time = pd.merge(df_fe, test2, on = 'ID')

In [30]:
df_fe_resolution_time.sort_values(by = 'TOTAL_TICKETS')

,ID,LEVEL_NAME,TOTAL_TICKETS,RES_TIME_TOTAL,RES_TIME_MEAN
97824,1499730,silver,1,125.89,125.89
24138,695719,rookie,1,96.84,96.84
77090,1345703,rookie,1,48.03,48.03
77092,1405979,rookie,1,19.44,19.44
77095,617040,rookie,1,11.60,11.60
...,...,...,...,...,...
4820,1359488,diamond,230,6103.86,26.54
6000,1243322,bronze,232,10349.56,44.61
985,342502,bronze,267,42996.66,161.04
492,773351,bronze,275,64370.34,234.07


In [31]:
df_fe_resolution_time = df_fe_resolution_time.reset_index(0)